# Time-Varying Beam Current — Current Profile Demo

Real cyclotron beams aren’t perfectly stable. Current can ramp up, fluctuate,
or follow deliberate schedules (e.g. pulsed irradiation). HYRR supports this
via `CurrentProfile` — a piecewise-constant current timeseries that modulates
the production rates during the coupled ODE solve.

We compare three scenarios on the standard **p + Mo-100 → Tc-99m** case:
1. **Constant current** (baseline)
2. **Triangular wave** (`/\\/\\/\\`) — periodic current cycling
3. **Triangular wave + noise** — realistic cyclotron jitter

## 1. Setup

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from hyrr import (
    Beam, DataStore, Layer, TargetStack,
    compute_stack, result_summary,
)
from hyrr.materials import resolve_element
from hyrr.models import CurrentProfile

db = DataStore("../nucl-parquet")

## 2. Define beam and target

Same setup as the quickstart: 16 MeV protons on enriched Mo-100.

In [ ]:
NOMINAL_CURRENT_MA = 0.15
IRR_TIME = 86400.0   # 24 h
COOL_TIME = 86400.0  # 24 h

beam = Beam(projectile="p", energy_MeV=16.0, current_mA=NOMINAL_CURRENT_MA)
mo100 = resolve_element(db, "Mo", enrichment={100: 1.0})

target = Layer(
    density_g_cm3=10.22,
    elements=[(mo100, 1.0)],
    energy_out_MeV=12.0,
)

print(f"Beam: {beam.projectile} at {beam.energy_MeV} MeV, {beam.current_mA} mA")
print(f"Irradiation: {IRR_TIME/3600:.0f} h, Cooling: {COOL_TIME/3600:.0f} h")

## 3. Build current profiles

We create three profiles with the **same average current** (0.15 mA) so
the differences come purely from the time structure:

- **Constant**: flat at 0.15 mA
- **Triangle**: ramps linearly between 0.05 and 0.25 mA in 4-hour cycles
- **Triangle + noise**: same triangle with ±10% Gaussian jitter

In [ ]:
rng = np.random.default_rng(42)

# Time grid: 1-minute steps over 24 hours
dt_step = 60.0  # 1 minute
times = np.arange(0, IRR_TIME, dt_step)

# --- Triangle wave ---
cycle_period = 4 * 3600  # 4-hour cycle
phase = (times % cycle_period) / cycle_period  # 0..1 sawtooth
triangle = np.where(phase < 0.5, 2 * phase, 2 * (1 - phase))  # 0..1..0 triangle

I_min, I_max = 0.05, 0.25  # mA
triangle_current = I_min + triangle * (I_max - I_min)

# --- Triangle + noise ---
noise_std = 0.10 * NOMINAL_CURRENT_MA  # 10% of nominal
noisy_current = triangle_current + rng.normal(0, noise_std, size=len(times))
noisy_current = np.clip(noisy_current, 0.0, None)  # no negative current

# Build CurrentProfile objects
cp_triangle = CurrentProfile(times_s=times, currents_mA=triangle_current)
cp_noisy = CurrentProfile(times_s=times, currents_mA=noisy_current)

print(f"Triangle: mean = {triangle_current.mean():.4f} mA, "
      f"min = {triangle_current.min():.4f}, max = {triangle_current.max():.4f}")
print(f"Noisy:    mean = {noisy_current.mean():.4f} mA, "
      f"min = {noisy_current.min():.4f}, max = {noisy_current.max():.4f}")

### Visualise the current profiles

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True, sharey=True)

t_hours = times / 3600

axes[0].plot(t_hours, np.full_like(times, NOMINAL_CURRENT_MA), "C0", lw=1.5)
axes[0].set_title("Constant")
axes[0].set_ylabel("I [mA]")

axes[1].plot(t_hours, triangle_current, "C1", lw=0.8)
axes[1].set_title("Triangle wave")
axes[1].set_ylabel("I [mA]")

axes[2].plot(t_hours, noisy_current, "C2", lw=0.4, alpha=0.7)
axes[2].plot(t_hours, triangle_current, "C1", lw=0.8, alpha=0.5, label="underlying triangle")
axes[2].set_title("Triangle + 10% Gaussian noise")
axes[2].set_ylabel("I [mA]")
axes[2].set_xlabel("Time [hours]")
axes[2].legend(fontsize=9)

for ax in axes:
    ax.axhline(NOMINAL_CURRENT_MA, color="k", ls="--", lw=0.6, alpha=0.4)
    ax.set_ylim(0, 0.35)

fig.tight_layout()
fig.savefig("/tmp/current_profiles.png", dpi=100)
plt.show()

## 4. Run simulations

Three runs: constant, triangle, and noisy triangle.

In [ ]:
def make_stack(profile=None):
    return TargetStack(
        beam=beam,
        layers=[target],
        irradiation_time_s=IRR_TIME,
        cooling_time_s=COOL_TIME,
        current_profile=profile,
    )

result_const = compute_stack(db, make_stack())
result_tri   = compute_stack(db, make_stack(cp_triangle))
result_noisy = compute_stack(db, make_stack(cp_noisy))

print("All three simulations complete.")

## 5. Compare Tc-99m activity curves

The main product of interest. How does time-varying current affect
activity build-up and decay?

In [ ]:
def get_tc99m(result):
    return result.layer_results[0].isotope_results["Tc-99m"]

tc_const = get_tc99m(result_const)
tc_tri   = get_tc99m(result_tri)
tc_noisy = get_tc99m(result_noisy)

fig, ax = plt.subplots(figsize=(10, 5))

for tc, label, color in [
    (tc_const, "Constant", "C0"),
    (tc_tri,   "Triangle", "C1"),
    (tc_noisy, "Triangle + noise", "C2"),
]:
    ax.plot(tc.time_grid_s / 3600, tc.activity_vs_time_Bq * 1e-9,
            label=label, color=color, lw=1.5)

ax.axvline(IRR_TIME / 3600, color="grey", ls=":", lw=1, label="EOI")
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Tc-99m activity [GBq]")
ax.set_title("Tc-99m activity: constant vs time-varying beam current")
ax.legend()
fig.tight_layout()
plt.show()

## 6. End-of-irradiation comparison table

In [ ]:
print(f"{'Scenario':<25s} {'EOI Tc-99m [GBq]':>18s} {'vs constant':>12s}  {'Source':>8s}")
print("-" * 68)

a_const = tc_const.activity_Bq

for label, tc in [
    ("Constant", tc_const),
    ("Triangle", tc_tri),
    ("Triangle + noise", tc_noisy),
]:
    a = tc.activity_Bq
    pct = (a / a_const - 1) * 100 if a_const > 0 else 0
    print(f"{label:<25s} {a*1e-9:>18.4f} {pct:>+11.2f}%  {tc.source:>8s}")

## 7. Source attribution: direct vs in-growth

The chain solver tracks whether activity comes from direct production
or daughter in-growth. Let’s compare for the triangle case.

In [ ]:
lr_tri = result_tri.layer_results[0]

fig, ax = plt.subplots(figsize=(10, 5))

for name, iso in sorted(
    lr_tri.isotope_results.items(),
    key=lambda kv: kv[1].activity_Bq, reverse=True,
)[:6]:
    t_h = iso.time_grid_s / 3600
    ax.plot(t_h, iso.activity_vs_time_Bq * 1e-9, label=f"{name} ({iso.source})", lw=1.5)

    # Show direct component as dashed if it differs from total
    if iso.source == "both" and len(iso.activity_direct_vs_time_Bq) > 0:
        ax.plot(t_h, iso.activity_direct_vs_time_Bq * 1e-9,
                ls="--", alpha=0.5, lw=1, color=ax.get_lines()[-1].get_color())

ax.axvline(IRR_TIME / 3600, color="grey", ls=":", lw=1)
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Activity [GBq]")
ax.set_title("Source attribution (triangle profile) — dashed = direct only")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 8. Full text summary (triangle profile)

In [ ]:
print(result_summary(result_tri))

## Key takeaways

- For the same **average** beam current, a time-varying profile produces
  slightly different EOI activity. The difference is typically small for
  isotopes with half-lives much longer or shorter than the cycle period,
  but can matter for isotopes whose half-life is comparable to the
  modulation timescale.
- The `CurrentProfile` is piecewise-constant: each entry specifies the
  current from that time until the next entry. Use 1-minute or finer
  steps for smooth profiles.
- Source attribution (`direct` / `daughter` / `both`) works correctly
  with time-varying current — the direct component uses the same
  piecewise stepping.